In [ ]:
import gym
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
import random
from collections import deque

# Set up the environment
env = gym.make("CartPole-v1")

# Set the hyperparameters
state_shape = (4,)  # State shape for CartPole: [cart position, cart velocity, pole angle, pole angular velocity]
action_space = env.action_space.n  # Number of possible actions (2 actions: left or right)
learning_rate = 0.001
gamma = 0.99  # Discount factor
epsilon = 0.1  # Exploration factor (epsilon-greedy)
batch_size = 64
memory = deque(maxlen=2000)  # Experience replay memory

# Build the neural network model
def build_model():
    model = models.Sequential()
    model.add(layers.Dense(24, input_dim=state_shape[0], activation='relu'))  # Fully connected layer
    model.add(layers.Dense(24, activation='relu'))  # Another hidden layer
    model.add(layers.Dense(action_space, activation='linear'))  # Output layer with action space size
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate), loss='mse')
    return model

# Epsilon-greedy policy for action selection
def select_action(state, model, epsilon):
    if random.random() < epsilon:
        return env.action_space.sample()  # Explore: Random action
    else:
        q_values = model.predict(state[np.newaxis])  # Predict Q-values for the given state
        return np.argmax(q_values[0])  # Exploit: Action with highest Q-value

# Experience replay
def replay(batch_size, model, target_model):
    if len(memory) < batch_size:
        return
    batch = random.sample(memory, batch_size)
    for state, action, reward, next_state, done in batch:
        target = reward
        if not done:
            target += gamma * np.max(target_model.predict(next_state[np.newaxis]))  # Predict Q-value for next state
        target_f = model.predict(state[np.newaxis])
        target_f[0][action] = target
        model.fit(state[np.newaxis], target_f, epochs=1, verbose=0)  # Train the model with updated target

# Training loop
episodes = 1000
model = build_model()  # Initialize the Q-network model
target_model = build_model()  # Initialize the target model
target_model.set_weights(model.get_weights())  # Copy initial weights

# Experience replay and training
for episode in range(episodes):
    state = env.reset()
    state, _ = env.reset()  # Reset the environment and get the state
    state = np.array(state)  # Convert state to numpy array (ensures it's in the correct shape)
    total_reward = 0
    done = False

    while not done:
        action = select_action(state, model, epsilon)  # Select action using epsilon-greedy policy
        next_state, reward, done, truncated, info = env.step(action)  # Take action and observe the next state
        next_state = np.array(next_state)  # Convert next state to numpy array

        # Store the experience in memory
        memory.append((state, action, reward, next_state, done))

        # Perform experience replay
        replay(batch_size, model, target_model)

        state = next_state  # Move to the next state
        total_reward += reward  # Accumulate the reward

    # Update the target model periodically
    if episode % 10 == 0:
        target_model.set_weights(model.get_weights())

    print(f"Episode {episode}/{episodes}, Total Reward: {total_reward}")

env.close()
